# 🏫 Teacher Efficiency Agent
### LangGraph + Gemini AI Pipeline

This notebook builds a **4-node LangGraph pipeline** that:
1. 📥 Ingests a teacher working-hours Excel file
2. 🤖 Calls Gemini LLM to evaluate each teacher's monthly efficiency *(Pydantic structured output)*
3. 📝 Calls Gemini LLM again to write an executive summary per month *(Pydantic structured output)*
4. 📊 Exports a fully formatted, color-coded Excel report

**Run every cell from top to bottom. Read the comments — they tell you what each cell does.**

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# Run once. If Colab asks to restart the runtime, click Restart
# then continue running from Cell 2.
# ════════════════════════════════════════════════════════
!pip install -q langgraph langchain-google-genai openpyxl pandas pydantic
print("✅ All packages installed")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — Set your Gemini API Key
#
# Get a FREE key (no credit card needed):
#   1. Open https://aistudio.google.com/app/apikey
#   2. Sign in with Google, click 'Create API Key'
#   3. Copy the key and paste it below
#
# Alternative — use Colab Secrets (recommended):
#   Click the 🔑 key icon in the left sidebar
#   Add a secret named GOOGLE_API_KEY with your key value
#   Then uncomment the two lines marked OPTION B below
# ════════════════════════════════════════════════════════

import os

# OPTION A — paste key directly (simple but visible in notebook)
GOOGLE_API_KEY = "YOUR_GEMINI_API_KEY_HERE"   # <- paste your key here

# OPTION B — read from Colab Secrets (safer, uncomment to use)
# from google.colab import userdata
# GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

if GOOGLE_API_KEY == "YOUR_GEMINI_API_KEY_HERE":
    raise ValueError("\n\nERROR: Please paste your Gemini API key in CELL 2 before continuing!\n")
print("✅ API key set")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — Upload your input Excel file
#
# A file picker will appear below.
# Select your  teacher_hours_input.xlsx  file.
# ════════════════════════════════════════════════════════

from google.colab import files

print("Please upload your teacher_hours_input.xlsx:")
uploaded = files.upload()

INPUT_FILE = list(uploaded.keys())[0]
OUTPUT_FILE = "teacher_efficiency_report.xlsx"

print(f"\nUploaded: {INPUT_FILE}")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — Pydantic Models  (equivalent to models.py)
# ════════════════════════════════════════════════════════

from pydantic import BaseModel, Field


class TeacherEfficiencyAnalysis(BaseModel):
    """LLM Call #1 output — per-teacher monthly efficiency."""
    efficiency_score: int = Field(..., ge=0, le=100)
    punctuality_rating: str
    reliability_rating: str
    workload_balance: str
    key_strengths: str
    areas_for_improvement: str
    recommendation: str


class MonthlyReport(BaseModel):
    """LLM Call #2 output — monthly executive summary."""
    executive_summary: str


class TeacherMonthStats(BaseModel):
    teacher_name: str
    month: int
    month_name: str
    subject: str
    total_sessions: int
    scheduled_hours: float
    actual_hours: float
    completion_rate: float
    replacements_given: int
    replacements_received: int
    sessions_with_notes: int
    classes_list: str


class AgentState(BaseModel):
    """LangGraph pipeline state — passed between all nodes."""
    input_file: str = ""
    raw_stats: list[dict] = Field(default_factory=list)
    efficiency_results: list[dict] = Field(default_factory=list)
    monthly_summaries: dict[str, str] = Field(default_factory=dict)
    output_file: str = ""
    errors: list[str] = Field(default_factory=list)


print("✅ Pydantic models defined")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — Prompts  (equivalent to prompts.py)
# ════════════════════════════════════════════════════════

EFFICIENCY_ANALYSIS_PROMPT = """
You are an expert school performance analyst. Analyze the following teacher's monthly
statistics and return a structured JSON efficiency evaluation.

Teacher: {teacher_name}
Month: {month_name}
Subject: {subject}

Statistics:
- Total Sessions: {total_sessions}
- Scheduled Hours: {scheduled_hours}
- Actual Hours: {actual_hours}
- Hours Completion Rate: {completion_rate:.1f}%
- Times teacher gave class to a replacement: {replacements_given}
- Times teacher received a replacement class: {replacements_received}
- Sessions with notes/issues: {sessions_with_notes}
- Classes taught: {classes_list}

Respond ONLY with a valid JSON object, no markdown, no explanation:
{{
  "efficiency_score": <integer 0-100>,
  "punctuality_rating": "<Excellent|Good|Average|Poor>",
  "reliability_rating": "<Excellent|Good|Average|Poor>",
  "workload_balance": "<Overloaded|Balanced|Underutilized>",
  "key_strengths": "<one concise sentence>",
  "areas_for_improvement": "<one concise sentence>",
  "recommendation": "<Continue|Monitor|Support Needed>"
}}
"""

SUMMARY_REPORT_PROMPT = """
You are a school principal writing a monthly summary for the teachers council.

Efficiency data for ALL teachers this month ({month_name}):

{teacher_summaries}

Write a brief (3-5 sentences) professional executive summary highlighting:
1. Overall department performance
2. Who stood out positively
3. Who may need support
4. One actionable recommendation for next month

Respond with plain text only, no bullet points, no markdown.
"""

print("✅ Prompts defined")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — Initialize the Gemini LLM
# ════════════════════════════════════════════════════════

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.2,
)

# Quick connection check
test = llm.invoke("Reply with just: OK")
print("✅ LLM connected:", test.content.strip())

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — Node 1: Ingest Excel & Compute Statistics
# ════════════════════════════════════════════════════════

import pandas as pd
import calendar


def node_ingest(state: AgentState) -> AgentState:
    print("📥 [Node 1] Loading and computing teacher statistics...")
    df = pd.read_excel(state.input_file, parse_dates=["Date"])
    df["Month"] = df["Date"].dt.month
    df["Month_Name"] = df["Date"].dt.month.apply(lambda m: calendar.month_name[m])

    stats_list = []
    for (teacher, month), group in df.groupby(["Teacher Name", "Month"]):
        month_name = group["Month_Name"].iloc[0]
        subject = group["Subject"].mode()[0]
        scheduled = group["Scheduled Hours"].sum()
        actual = group["Actual Hours"].sum()
        completion_rate = (actual / scheduled * 100) if scheduled > 0 else 0
        given = group["Replacement Given To"].notna().sum()
        received = group["Replacement Received From"].notna().sum()
        notes_count = group["Notes"].notna().sum()
        classes = ", ".join(sorted(group["Class"].unique()))

        stat = TeacherMonthStats(
            teacher_name=teacher, month=month, month_name=month_name,
            subject=subject, total_sessions=len(group),
            scheduled_hours=round(float(scheduled), 1),
            actual_hours=round(float(actual), 1),
            completion_rate=round(float(completion_rate), 1),
            replacements_given=int(given), replacements_received=int(received),
            sessions_with_notes=int(notes_count), classes_list=classes,
        )
        stats_list.append(stat.model_dump())

    state.raw_stats = stats_list
    print(f"   ✅ Computed stats for {len(stats_list)} teacher-month combinations.")
    return state


print("✅ Node 1 defined")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — Node 2: LLM Call #1 — Efficiency Analysis
# ════════════════════════════════════════════════════════

import json
from pydantic import ValidationError


def node_llm_efficiency(state: AgentState) -> AgentState:
    print("🤖 [Node 2] Running LLM efficiency analysis for each teacher-month...")
    results = []

    for stat in state.raw_stats:
        prompt = EFFICIENCY_ANALYSIS_PROMPT.format(**stat)
        try:
            response = llm.invoke(prompt)
            raw_text = response.content.strip()
            if raw_text.startswith("```"):
                raw_text = raw_text.split("```")[1]
                if raw_text.startswith("json"):
                    raw_text = raw_text[4:]
            parsed = TeacherEfficiencyAnalysis(**json.loads(raw_text))
            result = {**stat, **parsed.model_dump()}
        except Exception as e:
            print(f"   Warning: error for {stat['teacher_name']} / {stat['month_name']}: {e}")
            result = {
                **stat,
                "efficiency_score": None, "punctuality_rating": "N/A",
                "reliability_rating": "N/A", "workload_balance": "N/A",
                "key_strengths": "Analysis failed",
                "areas_for_improvement": "Analysis failed",
                "recommendation": "N/A",
            }
            state.errors.append(str(e))

        print(f"   ✅ {stat['teacher_name']} — {stat['month_name']}: score={result.get('efficiency_score')}")
        results.append(result)

    state.efficiency_results = results
    return state


print("✅ Node 2 defined")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 9 — Node 3: LLM Call #2 — Monthly Summary
# ════════════════════════════════════════════════════════

def node_llm_summary(state: AgentState) -> AgentState:
    print("📝 [Node 3] Generating monthly executive summaries...")

    by_month: dict[str, list[dict]] = {}
    for r in state.efficiency_results:
        by_month.setdefault(r["month_name"], []).append(r)

    summaries = {}
    for month_name, teachers in by_month.items():
        teacher_summaries = "\n".join(
            f"- {t['teacher_name']} ({t['subject']}): "
            f"score={t['efficiency_score']}, "
            f"completion={t['completion_rate']}%, "
            f"reliability={t['reliability_rating']}, "
            f"recommendation={t['recommendation']}"
            for t in teachers
        )
        prompt = SUMMARY_REPORT_PROMPT.format(
            month_name=month_name,
            teacher_summaries=teacher_summaries,
        )
        try:
            response = llm.invoke(prompt)
            summaries[month_name] = response.content.strip()
        except Exception as e:
            summaries[month_name] = f"Summary generation failed: {e}"
            state.errors.append(str(e))

        print(f"   ✅ Summary generated for {month_name}")

    state.monthly_summaries = summaries
    return state


print("✅ Node 3 defined")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 10 — Node 4: Export to Excel
# ════════════════════════════════════════════════════════

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


def node_export_excel(state: AgentState) -> AgentState:
    print("📊 [Node 4] Building output Excel report...")

    HEADER_BG = "1F3864"; HEADER_FG = "FFFFFF"
    ALT_ROW = "EAF0FB"; SCORE_HIGH = "C6EFCE"
    SCORE_MED = "FFEB9C"; SCORE_LOW = "FFC7CE"
    SUMMARY_BG = "D9E1F2"

    thin = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"), bottom=Side(style="thin"),
    )

    def header_style(cell, bg=HEADER_BG):
        cell.font = Font(name="Arial", bold=True, color=HEADER_FG, size=10)
        cell.fill = PatternFill("solid", start_color=bg)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = thin

    def data_style(cell, center=False, bg=None, bold=False):
        cell.font = Font(name="Arial", size=9, bold=bold)
        cell.alignment = Alignment(horizontal="center" if center else "left",
                                   vertical="center", wrap_text=True)
        if bg:
            cell.fill = PatternFill("solid", start_color=bg)
        cell.border = thin

    def score_color(s):
        return SCORE_HIGH if s >= 80 else SCORE_MED if s >= 60 else SCORE_LOW

    col_headers = [
        "Teacher", "Subject", "Sessions", "Sched.\nHrs", "Actual\nHrs",
        "Completion\n%", "Repl.\nGiven", "Repl.\nReceived", "Notes/\nIssues",
        "Efficiency\nScore", "Punctuality", "Reliability", "Workload",
        "Recommendation", "Key Strengths", "Areas for Improvement",
    ]
    col_widths = [20, 12, 10, 10, 10, 12, 10, 12, 10, 12, 13, 13, 14, 16, 38, 38]

    by_month = {}
    for r in state.efficiency_results:
        by_month.setdefault(r["month_name"], []).append(r)

    wb = Workbook()
    wb.remove(wb.active)
    all_rows = []

    for month_name, teachers in sorted(
        by_month.items(),
        key=lambda x: list(calendar.month_name).index(x[0])
    ):
        ws = wb.create_sheet(title=month_name)
        ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(col_headers))
        tc = ws.cell(row=1, column=1,
                     value=f"Teacher Efficiency Report — {month_name} 2024")
        tc.font = Font(name="Arial", bold=True, size=13, color="FFFFFF")
        tc.fill = PatternFill("solid", start_color="0D2137")
        tc.alignment = Alignment(horizontal="center", vertical="center")
        ws.row_dimensions[1].height = 24

        for ci, h in enumerate(col_headers, 1):
            header_style(ws.cell(row=2, column=ci, value=h))
        ws.row_dimensions[2].height = 34

        for ri, t in enumerate(sorted(teachers, key=lambda x: x['teacher_name']), 3):
            bg = ALT_ROW if ri % 2 == 0 else None
            vals = [
                t['teacher_name'], t['subject'], t['total_sessions'],
                t['scheduled_hours'], t['actual_hours'], f'=E{ri}/D{ri}',
                t['replacements_given'], t['replacements_received'],
                t['sessions_with_notes'], t['efficiency_score'],
                t['punctuality_rating'], t['reliability_rating'],
                t['workload_balance'], t['recommendation'],
                t['key_strengths'], t['areas_for_improvement'],
            ]
            for ci, val in enumerate(vals, 1):
                c = ws.cell(row=ri, column=ci, value=val)
                data_style(c, center=(ci not in [1, 15, 16]), bg=bg)
                if ci == 10 and t['efficiency_score']:
                    c.fill = PatternFill('solid',
                                        start_color=score_color(t['efficiency_score']))
                    c.font = Font(name='Arial', size=9, bold=True)
            ws.row_dimensions[ri].height = 42

        last = 2 + len(teachers)
        for r in range(3, last + 1):
            ws.cell(row=r, column=6).number_format = "0.0%"
        for i, w in enumerate(col_widths, 1):
            ws.column_dimensions[get_column_letter(i)].width = w

        # Executive summary box
        sr = last + 2
        ws.merge_cells(start_row=sr, start_column=1, end_row=sr, end_column=len(col_headers))
        lc = ws.cell(row=sr, column=1,
                     value=f"📋 Executive Summary — {month_name}")
        lc.font = Font(name="Arial", bold=True, size=10, color="1F3864")
        lc.fill = PatternFill("solid", start_color=SUMMARY_BG)
        lc.alignment = Alignment(horizontal="left", vertical="center")
        lc.border = thin

        tr = sr + 1
        ws.merge_cells(start_row=tr, start_column=1, end_row=tr + 2,
                       end_column=len(col_headers))
        sc = ws.cell(row=tr, column=1,
                     value=state.monthly_summaries.get(month_name, 'N/A'))
        sc.font = Font(name="Arial", size=9)
        sc.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
        sc.fill = PatternFill("solid", start_color="F2F6FC")
        sc.border = thin
        ws.row_dimensions[tr].height = 85
        ws.freeze_panes = "A3"
        all_rows.extend([(month_name, t) for t in teachers])

    # Overview sheet
    ov = wb.create_sheet(title="Overview", index=0)
    ov.merge_cells("A1:Q1")
    hd = ov.cell(row=1, column=1,
                 value="🏫 Teacher Efficiency — Full Overview (All Months)")
    hd.font = Font(name="Arial", bold=True, size=14, color="FFFFFF")
    hd.fill = PatternFill("solid", start_color="0D2137")
    hd.alignment = Alignment(horizontal="center", vertical="center")
    ov.row_dimensions[1].height = 28

    ov_headers = ["Month"] + col_headers
    ov_widths = [12] + col_widths
    for ci, h in enumerate(ov_headers, 1):
        header_style(ov.cell(row=2, column=ci, value=h))
    ov.row_dimensions[2].height = 34

    for ri, (month_name, t) in enumerate(all_rows, 3):
        bg = ALT_ROW if ri % 2 == 0 else None
        vals = [
            month_name, t['teacher_name'], t['subject'], t['total_sessions'],
            t['scheduled_hours'], t['actual_hours'], f'=F{ri}/E{ri}',
            t['replacements_given'], t['replacements_received'],
            t['sessions_with_notes'], t['efficiency_score'],
            t['punctuality_rating'], t['reliability_rating'],
            t['workload_balance'], t['recommendation'],
            t['key_strengths'], t['areas_for_improvement'],
        ]
        for ci, val in enumerate(vals, 1):
            c = ov.cell(row=ri, column=ci, value=val)
            data_style(c, center=(ci not in [2, 16, 17]), bg=bg)
            if ci == 11 and t['efficiency_score']:
                c.fill = PatternFill('solid',
                                    start_color=score_color(t['efficiency_score']))
                c.font = Font(name='Arial', size=9, bold=True)
        ov.row_dimensions[ri].height = 42

    last_ov = 2 + len(all_rows)
    for r in range(3, last_ov + 1):
        ov.cell(row=r, column=7).number_format = "0.0%"
    for i, w in enumerate(ov_widths, 1):
        ov.column_dimensions[get_column_letter(i)].width = w

    # Legend
    lr = last_ov + 2
    ov.cell(row=lr, column=1, value="Score Legend:").font = Font(
        name="Arial", bold=True, size=9)
    for ci, (lbl, col) in enumerate([
        ("Score 80+: High Performer", SCORE_HIGH),
        ("Score 60-79: Monitor", SCORE_MED),
        ("Score <60: Support Needed", SCORE_LOW),
    ], 2):
        c = ov.cell(row=lr, column=ci, value=lbl)
        c.fill = PatternFill("solid", start_color=col)
        c.font = Font(name="Arial", size=9)
        c.alignment = Alignment(horizontal="center")
        c.border = thin
    ov.freeze_panes = "A3"

    output_path = state.output_file or "teacher_efficiency_report.xlsx"
    wb.save(output_path)
    print(f"   ✅ Excel saved → {output_path}")
    state.output_file = output_path
    return state


print("✅ Node 4 defined")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 11 — Build & Compile the LangGraph Pipeline
# ════════════════════════════════════════════════════════

from langgraph.graph import StateGraph, END


def build_pipeline():
    graph = StateGraph(AgentState)
    graph.add_node("ingest",       node_ingest)
    graph.add_node("llm_analysis", node_llm_efficiency)
    graph.add_node("llm_summary",  node_llm_summary)
    graph.add_node("export",       node_export_excel)
    graph.set_entry_point("ingest")
    graph.add_edge("ingest",       "llm_analysis")
    graph.add_edge("llm_analysis", "llm_summary")
    graph.add_edge("llm_summary",  "export")
    graph.add_edge("export",       END)
    return graph.compile()


pipeline = build_pipeline()
print("✅ LangGraph pipeline compiled — 4 nodes ready")
print("   ingest → llm_analysis → llm_summary → export → END")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 12 — RUN THE FULL PIPELINE
# This is the main execution step. It will take ~1-2 minutes
# as it makes one LLM call per teacher per month.
# ════════════════════════════════════════════════════════

print("\n🚀 Starting Teacher Efficiency Agent Pipeline")
print("=" * 55)

initial_state = AgentState(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
)

final_state = pipeline.invoke(initial_state)

print("=" * 55)

if final_state['errors']:
    print(f"\n⚠️  {len(final_state['errors'])} error(s) during run:")
    for e in final_state['errors']:
        print(f"   • {e}")

print(f"\n✅ Pipeline complete! Report saved as: {final_state['output_file']}")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 13 — Download the Excel Report
# The file will save to your computer automatically.
# ════════════════════════════════════════════════════════

from google.colab import files

files.download(OUTPUT_FILE)
print(f"✅ Downloading {OUTPUT_FILE}...")